# 15 · Decision design for the specialist bank (Stage R5)

## Purpose
Stage R4 read "too thin to tell": the specialist bank separated withheld imbalance and vertical misalignment far better than
Mahalanobis, but its **normal specialist** accepted unfamiliar faults, so it called 92 withheld recordings healthy (19 for
Mahalanobis) and caught no more of them at a matched known-side rejection count. Autoencoders reconstruct inputs they never
trained on, most easily at low level (Zhou, CVPR 2022). This stage changes the **decision design**, not the specialists, and
re-scores R4's saved table (no training):
- **Step 0**: how many withheld recordings the arbitrator's own healthy gates (AE-6 ≤ τ_e and B1s ≤ τ_b) already pass — the
  floor for any design that lets those gates decide healthy.
- **E1**: the healthy gates decide healthy; recordings that fail them go to the **eight fault specialists only**.
- **E2**: a level correction s'_u = s_u − β L, with β = 1 (energy-relative error) or β fitted within class on known
  calibration recordings.

Declared in CLAUDE.md §7 (Stage R5) before running. Exploratory further analysis of development data, **seed 42 only**; the
reserved partitions stay sealed and F1 has already run, so nothing here is an independent claim. E1 was designed after
seeing R4's results. E3 (background model) is deferred to an addendum; E4 runs only after a pass and a new declaration.

## Inputs
`results/research/r4_ambiguity.csv` (R4's per-recording specialist scores), `results/scores/cnn6ae6_s42_loco_decisions.parquet`,
`results/scores/b1s_nomic_loco_q3fit_scores.parquet`, `results/loco/decision_thresholds.csv`, `metadata.csv`; the outputs
of `scripts/35_decision_design.py`: `results/research/r5_decision_design.json`, `r5_per_variant.csv`, `r5_per_recording.csv`.
Config and split hashes are printed below.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the configuration and metadata, prints the config and split hashes, and reads the R5 outputs; it checks that the per-recording table has all 11,548 rows of R4 and 779 withheld instances, one per distinct recording.</div>

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.config import load_config, config_sha256
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR
from ml.splits import FAULT_CLASSES, split_hash

ROOT = SYSTEM_CODE_DIR
R = ROOT / "results"
cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(ROOT / "metadata.csv")
print("config sha256:", config_sha256(cfg)[:16], "| split hash:", split_hash(meta)[:16])
res = json.loads((R / "research/r5_decision_design.json").read_text())
V = pd.read_csv(R / "research/r5_per_variant.csv")
P = pd.read_csv(R / "research/r5_per_recording.csv", keep_default_na=False, na_values=["", "nan", "NaN"])
un = P[P.role == "foldk_withheld"]
assert len(P) == 11548 and len(un) == 779 and un.rec_id.nunique() == 779
print("variants:", list(res["variants"]))

## Step 0 — the floor
Withheld recordings (779, one instance each) passing the arbitrator's healthy gates. The gates come from
`ml.arbitrator.decide` itself (`ml.specialists.healthy_gates`), after R0's tie rule. Any design in which those gates decide
healthy calls at least this many withheld faults healthy.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the floor (withheld recordings passing both healthy gates, with a Clopper–Pearson interval), how many pass each gate on its own, the floor per withheld class, and for context the healthy counts of the full decision rules.</div>

In [ ]:
s0 = res["step0_floor"]
print(f"floor: {s0['floor_gates_pass']} of {s0['withheld_instances']} ({s0['floor_rate']:.4f}, 95 % CI "
      f"{s0['floor_ci95'][0]:.4f}-{s0['floor_ci95'][1]:.4f}); AE gate alone {s0['ae_gate_pass']}, B1s gate alone {s0['b1s_gate_pass']}")
print("per class:", {c: f"{v['sum']}/{v['size']}" for c, v in s0["floor_by_class"].items()})
print("full rules, withheld called healthy:", {k: v["healthy"] for k, v in s0["full_rules_healthy"].items()})
print("known side passing the gates:", s0["known_side_gates_pass"])

## Every variant against its Mahalanobis baseline (primary: matched known-side rejection count)
Bank-composition variants (normal specialist kept, as R4) are compared with **M0** (R4's Mahalanobis-only); gated variants
with **M1** (Mahalanobis under the same healthy gates). At the matched count each variant calls exactly as many unseen-fold
known recordings unfamiliar as its baseline. The interval is a class-stratified bootstrap over the 779 withheld recordings
(sampling uncertainty; one seed, so there is no seed spread yet).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> tabulates, for every variant, the pooled unknown recall at the matched count against its baseline, the difference with its bootstrap interval, withheld recordings called healthy (the variant at the matched threshold and the baseline), the two acceptance criteria and the declared reading.</div>

In [ ]:
rows = []
for n, v in res["variants"].items():
    m = v["matched"]
    rows.append({"variant": n, "baseline": v["baseline"], "recall": m["variant_recall"], "baseline recall": m["baseline_recall"],
                 "Δ": m["diff"], "Δ 95% CI": f"[{m['paired_bootstrap']['ci95'][0]:+.3f}, {m['paired_bootstrap']['ci95'][1]:+.3f}]",
                 "healthy": v["withheld_healthy"]["variant_matched"], "baseline healthy": v["withheld_healthy"]["baseline"],
                 "+ambiguous healthy/fault": v["withheld_healthy"]["variant_ambiguous_healthy_fault_matched"],
                 "≥5 pp": v["criteria"]["recall_gain_ge_5pp"], "healthy ≤": v["criteria"]["healthy_not_higher"],
                 "reading": v["reading"]})
T = pd.DataFrame(rows).set_index("variant")
print(T.round(3).to_string())

## Threshold-free: AUROC and OSCR per fold
AUROC and OSCR of each variant's unfamiliarity score (known unseen-fold recordings vs the withheld class; OSCR correctness =
the CNN-6 top class), per outer fold, next to Mahalanobis on the same runs.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints, per variant and outer fold, the mean AUROC and OSCR over the fold's nine runs, with Mahalanobis alongside.</div>

In [ ]:
A = pd.concat({n: pd.DataFrame(v["auroc_oscr_per_fold"]).T for n, v in res["variants"].items()}, names=["variant", "fold"])
print(A.round(4).to_string())

## E1 per withheld class and at the mildest settings
A pooled gain can hide losses. Per withheld class: unknown recall at the matched count, E1 against M1, and the same at the
mildest setting of each family (bearings: 0 g added mass).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints E1's per-class recall at the matched count against M1, the per-class AUROC, and recall at the mildest setting of each family; then draws the per-class comparison (bars = pooled over folds, dots = each outer fold) and saves it.</div>

In [ ]:
e1 = res["variants"]["E1"]
pc = pd.DataFrame(e1["per_class"]).T
print(pc[["recall_matched", "baseline_recall", "recall_calibrated", "auroc", "maha_auroc", "healthy_matched", "n"]].round(3).to_string())
print(pd.DataFrame(e1["mildest"]).T.round(3).to_string())
v1 = V[V.variant == "E1"]
order = (pc.recall_matched - pc.baseline_recall).sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(11, 4.6))
xx = np.arange(len(order))
for off, col, lab, key in ((-0.2, "#4f6bff", "E1: healthy gates + 8 fault specialists", "unknown_recall_matched"),
                           (0.2, "#9e9e9e", "M1: healthy gates + Mahalanobis", "baseline_unknown_recall")):
    ax.bar(xx + off, [pc.loc[c, "recall_matched" if key == "unknown_recall_matched" else "baseline_recall"] for c in order],
           width=0.38, color=col, alpha=0.75, label=lab)
    for i, c in enumerate(order):
        g = v1[v1.withheld == c]
        ax.scatter(np.full(len(g), i + off), g[key] / g.n_unknown, color="k", s=10, zorder=3)
ax.set_xticks(xx, [c.replace("_", " ") for c in order], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("withheld recordings called unfamiliar (share)"); ax.set_ylim(0, 1.05)
ax.set_title("E1 vs M1 at the same known-side rejection count, per withheld class (dots = outer folds; seed 42)", fontsize=10)
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout(); fig.savefig(R / "figures/r5_e1_per_class.png", dpi=150); plt.show()

## What E1 costs on the known side (not part of the bar)
The bar looks at unfamiliar recordings. E1 also changes what happens to known ones: a known **normal** that fails the healthy
gates now goes to the fault-only bank, which can name a fault on a healthy machine; M1 calls such a recording unknown. And the
bank names known faults less reliably than the CNN-6.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints, for R4 and E1 against their baselines, how known normals are called (healthy, a named or ambiguous fault, unfamiliar) and how known faults are named (correct single specialist or CNN-6 top class, an ambiguous set containing the right class, wrong, healthy, unfamiliar).</div>

In [ ]:
for n in ("R4", "E1"):
    ks = res["variants"][n]["known_side"]
    print(f"--- {n} (baseline {res['variants'][n]['baseline']}), normals n={ks['normals']['n']}")
    print("  variant :", {k: round(v, 3) for k, v in ks["normals"]["variant"].items()})
    print("  baseline:", {k: round(v, 3) for k, v in ks["normals"]["baseline"].items()})
    print(f"  known faults n={ks['known_faults']['n']}:", {k: (round(v, 3) if isinstance(v, float) else v)
                                                         for k, v in ks["known_faults"].items() if k != "note"})

## E2 — the level correction
L = the cached dB level in natural-log power units; β = 1 gives the energy-relative error, E2b fits β within class on each
run's known calibration recordings. Stage R1 showed that removing a physical trend can remove signal; imbalance and the ball
faults are the loudest conditions, so they are called out.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the fitted β values, E2's per-class recall at the matched count next to its baseline for both compositions, and draws (left) the bank score against level for known and withheld recordings before and after the fitted correction, and (right) per-class recall of R4, E2a and E2b in the bank composition against M0. The figure is saved for the report.</div>

In [ ]:
for n in ("E2a-bank", "E2b-bank", "E1+E2a", "E1+E2b"):
    v = res["variants"][n]
    print(n, "β:", {k: round(x, 3) for k, x in v["beta"].items()}, "| recall", round(v["matched"]["variant_recall"], 3),
          "vs", round(v["matched"]["baseline_recall"], 3))
cmp = pd.DataFrame({n: pd.DataFrame(res["variants"][n]["per_class"]).T.recall_matched for n in ("R4", "E2a-bank", "E2b-bank", "E1", "E1+E2a", "E1+E2b")})
cmp["M0"] = pd.DataFrame(res["variants"]["R4"]["per_class"]).T.baseline_recall
cmp["M1"] = pd.DataFrame(res["variants"]["E1"]["per_class"]).T.baseline_recall
print(cmp.round(3).to_string())
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]
kn = P[P.role == "foldk_known"]
for frame, col, lab in ((kn, "#9e9e9e", "known (unseen fold)"), (un, "#ff8c00", "withheld")):
    ax.scatter(frame.level, frame["R4__S"], s=4, alpha=0.25, color=col, label=f"{lab}: R4 score")
    ax.scatter(frame.level, frame["E2b-bank__S"], s=4, alpha=0.25, color=col, marker="x")
ax.set_xlabel("cached input level (dB, vibration channels)"); ax.set_ylabel("bank score S (dots R4; crosses E2b-corrected)")
ax.set_title("Bank score vs input level", fontsize=10); ax.legend(fontsize=7, markerscale=3)
ax = axes[1]
order = list(FAULT_CLASSES)
xx = np.arange(len(order))
for j, (n, col) in enumerate((("M0", "#9e9e9e"), ("R4", "#4f6bff"), ("E2a-bank", "#ffb74d"), ("E2b-bank", "#e57373"))):
    ax.bar(xx + (j - 1.5) * 0.2, cmp.loc[order, n], width=0.2, color=col, label=n)
ax.set_xticks(xx, [c.replace("_", " ") for c in order], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("withheld called unfamiliar, matched count"); ax.set_ylim(0, 1.05)
ax.set_title("Level correction per withheld class (bank composition vs M0)", fontsize=10); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(R / "figures/r5_level_correction.png", dpi=150); plt.show()

## Calibrated thresholds, sensitivity and cost
Each variant at its own calibrated τ (97.5 % order statistic over the run's calibration recordings), not matched; for gated
variants also τ from known-fault calibration recordings only (declared sensitivity). CPU milliseconds per recording.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints each variant's unknown recall and known-side rejection at its own calibrated threshold with the baseline's known-side rejection, the gated sensitivity, and the CPU latency per recording (autoencoders vs CNN-6 + Mahalanobis).</div>

In [ ]:
for n, v in res["variants"].items():
    c = v["calibrated"]
    extra = v.get("sensitivity_tau_known_fault_cal_only")
    print(f"{n:9s} calibrated: recall {c['variant_recall']:.3f}, known rejection {c['variant_known_rejection']:.3f} "
          f"(baseline {c['baseline_known_rejection']:.3f}), withheld healthy {c['withheld_healthy']}"
          + (f" | fault-cal τ: recall {extra['variant_recall']:.3f}, known rejection {extra['variant_known_rejection']:.3f}" if extra else ""))
print(res["latency"]["note"]); print({t: {k: round(x, 1) for k, x in d.items()} for t, d in res["latency"]["cpu_ms_per_recording"].items()})

## Findings

Exploratory further analysis of development data, **seed 42 only**; E1 was designed after seeing R4. Reserved partitions
sealed; F1 already run; nothing here is an independent claim.

- **Step 0, the floor is tiny: 3 of 779** withheld recordings pass both healthy gates (0.4 %, 95 % CI 0.08–1.1 %; AE gate
  alone 18, B1s gate alone 29; 2 underhang cage, 1 vertical misalignment). The full rules call 2 (D) to 28 (A) healthy, B and
  B_with_caution 19. So letting the gates decide healthy costs almost nothing on unfamiliar faults — but 32 % of known
  normals also fail the gates (237 of 6,457 known instances pass; 67.6 % of known normals).
- **E1 passes the declared bar at seed 42.** Healthy gates + the eight fault specialists vs Mahalanobis under the same gates
  (M1), matched known-side rejection count: pooled unknown recall **0.733 vs 0.602 (+13.1 points, class-stratified bootstrap
  +9.8 to +16.4)**; withheld called healthy **3 vs 3** (equal by construction: both equal the floor). Per fold 0.891 vs 0.647,
  0.623 vs 0.608, 0.690 vs 0.548; AUROC 0.967 / 0.969 / 0.925 vs 0.870 / 0.893 / 0.868. At its own calibrated τ: recall 0.737
  at known rejection 0.088 (M1 0.099). Removing the normal specialist is what cures R4's failure: R4 at the matched count
  called 98 withheld recordings healthy (and 142 more "healthy or fault").
- **The gain is partial and fold-dependent.** Gains: imbalance 0.444 vs 0.035, vertical misalignment 0.600 vs 0.080, ball
  faults 1.000 vs 0.84–0.86, mild overhang cage 0.952 vs 0.571. Losses: overhang outer 0.608 vs 0.960, underhang cage 0.764 vs
  0.903, horizontal misalignment 0.890 vs 1.000; at the mildest settings underhang cage 0 g 0.480 vs 1.000 and overhang outer 0 g
  0.808 vs 1.000, and imbalance 6 g stays 0.000. Vertical misalignment is all-or-nothing by fold (1.00 / 0.00 / 1.00; fold 1
  holds 50 of its 125 recordings) and imbalance falls 0.80 / 0.46 / 0.06, so the pair gain is not stable across folds.
- **E1's known-side cost (outside the bar):** 22.7 % of known normals are named a fault (14.7 % one fault, 8.0 % an ambiguous
  set) where M1 calls those gate failures unknown (32.4 %); and the fault bank names known faults correctly on its own in only
  61.8 % of instances (+17.3 % ambiguous sets containing the right class) against 89.3 % for the CNN-6 top class. The bank is a
  useful unfamiliarity score, not a namer: any use would keep the CNN-6 for naming.
- **E2 (level correction) hurts in every form, as R1 warned.** Matched recall vs baseline: E2a-bank 0.406 vs 0.582, E2b-bank
  0.368, E1+E2a 0.511 vs 0.602, E1+E2b 0.425 (all "does not help"); AUROC falls to 0.69–0.88. β = 1 erases the loud classes'
  detectability (overhang ball 1.000 → 0.029, underhang ball 1.000 → 0.065). The fitted within-class β is **negative**
  (−0.11 to −0.35, mean −0.23): within a condition louder recordings are reconstructed relatively better, so R4's +0.60 score–
  level association is a between-class effect, and subtracting level removes class information rather than an artefact.
- **Cost:** no training; every variant still needs 9 autoencoder passes per recording (E1: AE-6 for the gate + 8 fault
  specialists), 105–119 ms on the CPU against 4–14 ms for CNN-6 + Mahalanobis. **Next, per the stage's seed policy:** E1 goes to
  seeds 43 / 44 only with Bhuvesh's approval (≈ 76 min of specialist training per seed); E3 awaits its addendum; E4 is gated
  on E1 holding up.


## Outputs written
- By `scripts/35_decision_design.py`: `results/research/r5_decision_design.json`, `r5_per_variant.csv`, `r5_per_recording.csv`.
- By this notebook: `results/figures/r5_e1_per_class.png`, `results/figures/r5_level_correction.png`.
- E3's `r5_background_difference.png` is not produced: E3 is deferred to an addendum.